In [5]:
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName("SparK first")
    .master("local[*]")
    .getOrCreate()
)
spark

In [11]:
rides_df=spark.read.csv("C:\\Users\\Administrator\\Downloads\\uber_rides.csv",header=True,inferSchema=True)

In [12]:
rides_df.show(5)
rides_df.printSchema()

+--------------+--------------+-------------+----------+---+---------+---------------+-----------+---------------+-----+-----------+-------------+---------------+-----+------+--------+--------+-----+-------------+---------+---------+---------+----------+----------+-----------+--------------+---------------+-------------+--------------+------------+--------+----------+---------+-------------+
|    Start_time|      End_time|customer_name|    Mobile|Age|Pin-Codes|         Source|Vaccine_cus|    Destination|Miles|Est_Costing|Ride_category|        Purpose| temp|clouds|pressure|humidity| wind|accquire_vehi|free_vehi|Lattitute|Longitude|locationID|rating_cus|Driver_Name|Driver_contact|Trusted_Contact|Driver_rating|Vaccine_Driver|Payment_mode|Discount|Final_cost|   Status|Trip Distance|
+--------------+--------------+-------------+----------+---+---------+---------------+-----------+---------------+-----+-----------+-------------+---------------+-----+------+--------+--------+-----+-----------

# Q1.top 5 drivers from dataset

In [14]:
from pyspark.sql.functions import avg,col,count,dense_rank,sum,max,min
from pyspark.sql.types import FloatType
from pyspark.sql.window import Window

In [21]:
rides_df.groupBy("Driver_Name").agg(avg(col("Driver_rating")).alias("Performance")).orderBy("Performance",ascending=False).show()

+-----------+-----------+
|Driver_Name|Performance|
+-----------+-----------+
|     Mikkel|        5.0|
|      Sandy|        5.0|
|      Marta|        5.0|
|        Kip|        5.0|
|      Myrle|        5.0|
|       Kain|        5.0|
|       Burk|        5.0|
|        Tim|        5.0|
|      Cindi|        5.0|
|       Dora|        5.0|
|      Margi|        4.9|
|      Tonie|        4.9|
|      Jacky|        4.9|
|      Silas|        4.9|
|     Shaine|        4.9|
|      Robby|        4.9|
|      Daron|        4.9|
|     Sybila|        4.9|
|     Luelle|        4.9|
|   Rosmunda|        4.9|
+-----------+-----------+
only showing top 20 rows



# Q2.driver who have most number of cancelled trips

In [30]:
rides_df.filter(col("Status")=="Cancelled").select("Driver_Name","Status").groupBy("Driver_Name","Status")\
.agg(count(col("Driver_Name")).alias("Total_Count"))\
.orderBy("Total_Count",ascending=False).show(5)

+-----------+---------+-----------+
|Driver_Name|   Status|Total_Count|
+-----------+---------+-----------+
|  Katherina|Cancelled|          3|
|   Costanza|Cancelled|          2|
|     Lebbie|Cancelled|          2|
|     Sherry|Cancelled|          2|
|       Burk|Cancelled|          2|
+-----------+---------+-----------+
only showing top 5 rows



In [32]:
rides_df.select("Driver_Name","Status").where(col("Status")=="Cancelled")\
.groupBy("Driver_Name","Status").agg(count(col("Driver_Name")).alias("Total_count")).orderBy("Total_count",ascending=False).show(5)

+-----------+---------+-----------+
|Driver_Name|   Status|Total_count|
+-----------+---------+-----------+
|  Katherina|Cancelled|          3|
|   Costanza|Cancelled|          2|
|     Lebbie|Cancelled|          2|
|     Sherry|Cancelled|          2|
|       Burk|Cancelled|          2|
+-----------+---------+-----------+
only showing top 5 rows



# Q3. most prefered ride_category from dataset

In [34]:
rides_df.select("Ride_category").groupBy("Ride_category").agg(count("*").alias("Total_count")).orderBy("Total_count",ascending=False).show()

+-------------+-----------+
|Ride_category|Total_count|
+-------------+-----------+
|         Bike|        100|
|        Prime|        100|
|         Auto|        100|
|    Uber-Mini|        100|
|   Uber-Micro|        100|
+-------------+-----------+



# Q4. driver who got highest amount of revenue

In [7]:
from pyspark.sql.types import FloatType

rides_df = rides_df.withColumn("Final_cost", col("Final_cost").cast(FloatType()))

In [11]:



highest_revenue_df=rides_df.groupBy("Driver_Name")\
    .agg(
        sum("Final_cost").alias("total_revenue")
    )\
    .orderBy("total_revenue", ascending=False)
highest_revenue_df.show(1)

+-----------+-----------------+
|Driver_Name|    total_revenue|
+-----------+-----------------+
|     Aurlie|5787.095993041992|
+-----------+-----------------+
only showing top 1 row



# Q5. second highest rated driver

In [25]:
avg_df = rides_df.groupBy("Driver_Name").agg(avg("Driver_rating").alias("avg_rating"))
window_spec = Window.orderBy(avg_df["avg_rating"].desc())
ranked_df = avg_df.withColumn("rank", dense_rank().over(window_spec))
second_highest_df = ranked_df.filter(col("rank") == 2).select("Driver_Name","avg_rating").show(5)

+-----------+----------+
|Driver_Name|avg_rating|
+-----------+----------+
|      Margi|       4.9|
|      Tonie|       4.9|
|      Jacky|       4.9|
|      Silas|       4.9|
|     Shaine|       4.9|
+-----------+----------+
only showing top 5 rows



# Q6 total revenue of top five drivers

In [45]:
from pyspark.sql.functions import avg, sum, col

total_revenue_df = rides_df.groupBy("Driver_Name")\
    .agg(
        avg("Driver_rating").alias("avg_rating"),
        sum("Final_cost").alias("total_revenue")
    )\
    .orderBy(
        col("avg_rating").desc(),
        col("total_revenue").desc()
    )

total_revenue_df.show(5)

+-----------+----------+------------------+
|Driver_Name|avg_rating|     total_revenue|
+-----------+----------+------------------+
|        Kip|       5.0| 1355.999984741211|
|     Mikkel|       5.0|1249.3439989089966|
|      Cindi|       5.0| 1083.379997253418|
|       Dora|       5.0| 902.3400001525879|
|        Tim|       5.0| 883.8500061035156|
+-----------+----------+------------------+
only showing top 5 rows



# Q7. source which generated most amount of revenue

In [49]:
source_wise_revenue=rides_df.groupBy("Source")\
.agg(
    sum("Final_cost").alias("Total_revenue")
).orderBy(
    col("Total_revenue").desc()
)
source_wise_revenue.show(1)

+-----------+------------------+
|     Source|     Total_revenue|
+-----------+------------------+
|Fort Pierce|24419.943781852722|
+-----------+------------------+
only showing top 1 row



# Q8. Source which have most number of cancelled trips

In [57]:
most_cancelled_source_df=rides_df.filter(col("Status")=="Cancelled").groupBy("Source").agg(count("Source").alias("Total_cancellations"))\
.orderBy(col("Total_cancellations").desc())

most_cancelled_source_df.show(1)

+-----------+-------------------+
|     Source|Total_cancellations|
+-----------+-------------------+
|Fort Pierce|                 13|
+-----------+-------------------+
only showing top 1 row



# Q9. Ride category which generate most amount of revenue

In [60]:
ride_category_revenue=rides_df.groupBy("Ride_category").agg(
    sum("Final_cost").alias("Total_revenue")
).orderBy(
    col("Total_revenue").desc()
)

ride_category_revenue.show()

+-------------+------------------+
|Ride_category|     Total_revenue|
+-------------+------------------+
|        Prime|29717.699966430664|
|   Uber-Micro|          21607.25|
|         Auto|19870.949971199036|
|    Uber-Mini|14452.600006103516|
|         Bike|13278.509815692902|
+-------------+------------------+



# Q10. maximum and minimum humidity of the city during completed rides

In [19]:
max_min_df = rides_df.filter(col("Status") == "Completed").agg(
    max(col("humidity")).alias("Maximum_Humidity"),
    min(col("humidity")).alias("Minimum_Humidity")
)

max_min_df.show()

+----------------+----------------+
|Maximum_Humidity|Minimum_Humidity|
+----------------+----------------+
|            0.95|            0.45|
+----------------+----------------+

